In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import metpy.calc as mpcalc
from metpy.units import units
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from pathlib import Path
from tqdm.auto import tqdm


In [ ]:
# Variant helper: plot any background field, show all 925 hPa GPH contours in grey,
# and highlight one configurable GPH contour level.

def plot_background_with_gph_contour_highlight(
    extrema_context,
    background_frames,
    field_name,
    colorbar_label,
    cmap_name="Spectral_r",
    bin_width=1.0,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
):
    frame_rows = extrema_context["frame_rows"]
    trajectory_line = extrema_context["trajectory_line"]
    gph_frames = extrema_context["gph_frames"]
    records = extrema_context["records"]

    if "frame" not in background_frames.dims:
        raise RuntimeError("background_frames must have a 'frame' dimension.")
    if background_frames.sizes["frame"] != len(records):
        raise RuntimeError("background_frames frame count must match extrema records.")

    line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
    line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

    extent_lon_min = float(extrema_context["extent_lon_min"])
    extent_lon_max = float(extrema_context["extent_lon_max"])
    extent_lat_min = float(extrema_context["extent_lat_min"])
    extent_lat_max = float(extrema_context["extent_lat_max"])

    levels, cmap, norm = _build_discrete_colormap(background_frames, bin_width, cmap_name)

    spacing_label = (
        f"{int(gph_contour_spacing_m)}"
        if float(gph_contour_spacing_m).is_integer()
        else f"{gph_contour_spacing_m:g}"
    )
    highlight_label = (
        f"{int(highlight_gph_level_m)}"
        if float(highlight_gph_level_m).is_integer()
        else f"{highlight_gph_level_m:g}"
    )

    gph_vmin = float(np.floor(float(gph_frames.min().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
    gph_vmax = float(np.ceil(float(gph_frames.max().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
    gph_contour_levels = np.arange(gph_vmin, gph_vmax + gph_contour_spacing_m, gph_contour_spacing_m)
    base_gph_levels = gph_contour_levels[~np.isclose(gph_contour_levels, highlight_gph_level_m)]

    for i, row in frame_rows.iterrows():
        record = records[i]
        frame_ts = pd.Timestamp(row["valid_time"]).round("h")
        frame_step = row.get("step_hour", np.nan)

        background_frame = background_frames.isel(frame=i)
        gph_frame = gph_frames.isel(frame=i)

        lon_curr = float(row["longitude_360"])
        lat_curr = float(row["latitude"])

        long_end_a = np.asarray(record["long_end_a"], dtype=float)
        long_end_b = np.asarray(record["long_end_b"], dtype=float)

        fig = plt.figure(figsize=(14, 7))
        ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

        ax.coastlines(resolution="110m", linewidth=0.8, color="black")
        ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

        mesh = ax.pcolormesh(
            background_frame["longitude"],
            background_frame["latitude"],
            background_frame,
            cmap=cmap,
            norm=norm,
            transform=ccrs.PlateCarree(),
            shading="auto",
            zorder=1,
        )

        if base_gph_levels.size > 0:
            contour_gph = ax.contour(
                gph_frame["longitude"],
                gph_frame["latitude"],
                gph_frame,
                levels=base_gph_levels,
                colors=contour_color,
                linewidths=0.8,
                alpha=0.95,
                transform=ccrs.PlateCarree(),
                zorder=4,
            )
            ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)

        highlight_contour_handle = None
        gph_frame_min = float(gph_frame.min().values)
        gph_frame_max = float(gph_frame.max().values)
        if gph_frame_min <= highlight_gph_level_m <= gph_frame_max:
            contour_highlight = ax.contour(
                gph_frame["longitude"],
                gph_frame["latitude"],
                gph_frame,
                levels=[highlight_gph_level_m],
                colors=highlight_color,
                linewidths=1.8,
                alpha=1.0,
                transform=ccrs.PlateCarree(),
                zorder=8,
            )
            ax.clabel(
                contour_highlight,
                contour_highlight.levels,
                fmt="%.0f",
                inline=True,
                fontsize=8,
                colors=highlight_color,
            )
            highlight_contour_handle = Line2D([0], [0], color=highlight_color, lw=1.8)

        (between_extrema_handle,) = ax.plot(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            color="magenta",
            linewidth=2.0,
            linestyle="-",
            transform=ccrs.PlateCarree(),
            zorder=9,
            label="Line between extrema stops",
        )

        extrema_endpoints_handle = ax.scatter(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            s=42,
            c="magenta",
            marker="x",
            linewidths=1.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label="Extrema stops",
        )

        (traj_line_handle,) = ax.plot(
            line_lons,
            line_lats,
            color="white",
            linewidth=1.8,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label="Backward trajectory (72h to 0h)",
        )

        traj_points_handle = ax.scatter(
            line_lons,
            line_lats,
            s=18,
            c="black",
            alpha=0.35,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label=TRAJECTORY_POINT_LABEL,
        )

        current_handle = ax.scatter(
            [lon_curr],
            [lat_curr],
            marker="x",
            s=130,
            c="deepskyblue",
            linewidths=2.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label=f"Current {extrema_context['frame_step_hours']}h point",
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False

        step_text = (
            f"t-{int(frame_step)}h"
            if pd.notna(frame_step)
            else f"{extrema_context['frame_step_hours']}h sample"
        )
        ax.set_title(
            f"{field_name} + 925 hPa GPH contours ({spacing_label} m spacing, {highlight_label} m highlighted) | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
        )

        cbar = plt.colorbar(
            mesh,
            ax=ax,
            boundaries=levels,
            orientation="horizontal",
            pad=0.05,
            shrink=0.9,
        )
        cbar.set_label(colorbar_label)

        contour_handle = Line2D([0], [0], color=contour_color, lw=0.9)
        legend_handles = [
            contour_handle,
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ]
        legend_labels = [
            f"925 hPa GPH contours ({spacing_label} m)",
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {extrema_context['frame_step_hours']}h point",
            "Line between extrema stops",
            "Extrema stops",
        ]
        if highlight_contour_handle is not None:
            legend_handles.append(highlight_contour_handle)
            legend_labels.append(f"{highlight_label} m GPH contour")

        ax.legend(legend_handles, legend_labels, loc="lower left")
        plt.tight_layout()
        plt.show()


In [ ]:
# Shared setup copied from causes-of-gph-extremas.ipynb, wrapped for reuse.
GPH_FILE = "era5_2021-nov_250-500-925_uv_pv_gph.nc"
TEMPERATURE_FILE = "era5_2021-nov_250-500-925_temperature.nc"
SPECIFIC_HUMIDITY_FILE = "specifichumidity_wind_1000-925hpa_2026-02-19.nc"

GPH_REGION_LON_MIN = 120.0
GPH_REGION_LON_MAX = 265.0
GPH_REGION_LAT_MIN = 30.0
GPH_REGION_LAT_MAX = 72.0
GPH_VMIN = 600.0
GPH_VMAX = 900.0

# These are used by _trace_gradient_path domain checks.
region_lon_min = GPH_REGION_LON_MIN
region_lon_max = GPH_REGION_LON_MAX
region_lat_min = 23.0
region_lat_max = GPH_REGION_LAT_MAX

gph_contour_levels = np.arange(560.0, 941.0, 20.0)


def resolve_data_path(filename: str) -> Path:
    candidates = [
        Path("../data") / filename,  # when kernel cwd is scripts/
        Path("data") / filename,     # when kernel cwd is repo root
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        checked = ", ".join(str(p) for p in candidates)
        raise FileNotFoundError(f"Could not find {filename}. Checked: {checked}")
    return path


def open_dataset_from_data_dir(filename: str) -> xr.Dataset:
    return xr.open_dataset(resolve_data_path(filename))


def load_gph_925_context(gph_filename: str = GPH_FILE):
    gph_path = resolve_data_path(gph_filename)
    ds_local = xr.open_dataset(gph_path)

    if "z" not in ds_local.data_vars:
        raise RuntimeError("Expected variable 'z' in GPH dataset.")
    if "pressure_level" not in ds_local.coords:
        raise RuntimeError("Expected pressure_level coordinate in GPH dataset.")

    z_925_raw = ds_local["z"].sel(pressure_level=925)
    gph_925_m = xr.apply_ufunc(np.divide, z_925_raw, 9.80665)
    gph_925_m.name = "gph_925_m"

    return {
        "gph_dataset": ds_local,
        "gph_925_m_all": gph_925_m,
        "gph_path": gph_path,
    }


gph_context = load_gph_925_context()
ds = gph_context["gph_dataset"]
gph_source_dataset = ds
GPH_ERA5_PATH = str(gph_context["gph_path"])
gph_925_m_all = gph_context["gph_925_m_all"]

gph_925_m_all


In [ ]:
# Backward trajectory function copied from causes-of-gph-extremas.ipynb.
VANCOUVER_LAT = 49.28
VANCOUVER_LON = -123.12
VANCOUVER_LON_360 = VANCOUVER_LON % 360.0
TRAJECTORY_START_TIME_UTC = pd.Timestamp("2021-11-01 00:00:00")
TRAJECTORY_END_TIME_UTC = pd.Timestamp("2021-11-12 15:00:00")
TRAJECTORY_TIME_STEP_HOURS = 6


def backward_integrate_trajectory_uv(
    ds,
    start_lat,
    start_lon,
    start_time,
    pressure_level=925,
    hours_back=None,
    time_step_hours=1,
    substeps=4,
    earth_radius_m=6_371_000.0,
):
    """Backward trajectory using ERA5 u/v winds sampled on a coarser time grid."""
    if hours_back is None:
        raise ValueError("hours_back must be provided.")
    hours_back = int(hours_back)
    if hours_back < 0:
        raise ValueError("hours_back must be >= 0")

    time_step_hours = int(time_step_hours)
    if time_step_hours < 1:
        raise ValueError("time_step_hours must be >= 1")

    if substeps < 1:
        raise ValueError("substeps must be >= 1")

    ds_uv = ds[["u", "v"]].sel(pressure_level=pressure_level)
    t0 = pd.Timestamp(start_time).to_datetime64()
    t_nearest = pd.Timestamp(
        ds_uv["valid_time"].sel(valid_time=t0, method="nearest").values
    ).round("h")

    step_hours = []
    remaining_hours = hours_back
    while remaining_hours > 0:
        step_hour = min(time_step_hours, remaining_hours)
        step_hours.append(step_hour)
        remaining_hours -= step_hour

    boundary_times = [t_nearest]
    for step_hour in step_hours:
        boundary_times.append(boundary_times[-1] - pd.Timedelta(hours=step_hour))

    sample_times = np.array(
        sorted({ts.to_datetime64() for ts in boundary_times}),
        dtype="datetime64[ns]",
    )
    ds_uv = ds_uv.sel(valid_time=sample_times).load()

    time_values_ns = ds_uv["valid_time"].values.astype("datetime64[ns]").astype(np.int64)
    lat_values = np.asarray(ds_uv["latitude"].values, dtype=np.float64)
    lon_values = np.mod(np.asarray(ds_uv["longitude"].values, dtype=np.float64), 360.0)
    u_values = np.asarray(ds_uv["u"].values)
    v_values = np.asarray(ds_uv["v"].values)

    if lat_values[0] > lat_values[-1]:
        lat_values = lat_values[::-1].copy()
        u_values = np.ascontiguousarray(u_values[:, ::-1, :])
        v_values = np.ascontiguousarray(v_values[:, ::-1, :])

    del ds_uv

    def _interp_axis(values, target):
        if values.size == 1:
            return 0, 0, 0.0

        upper = np.searchsorted(values, target, side="right")
        if upper <= 0:
            return 0, 0, 0.0
        if upper >= values.size:
            last = values.size - 1
            return last, last, 0.0

        lower = upper - 1
        span = values[upper] - values[lower]
        weight = 0.0 if span == 0 else float((target - values[lower]) / span)
        return lower, upper, weight

    def _interp_lon(lon_target):
        if lon_values.size == 1:
            return 0, 0, 0.0

        lon_target = float(lon_target) % 360.0
        upper = np.searchsorted(lon_values, lon_target, side="right") % lon_values.size
        lower = (upper - 1) % lon_values.size

        lon_lower = lon_values[lower]
        lon_upper = lon_values[upper] + (360.0 if upper <= lower else 0.0)
        lon_eval = lon_target + (360.0 if lon_target < lon_lower else 0.0)

        span = lon_upper - lon_lower
        weight = 0.0 if span == 0 else float((lon_eval - lon_lower) / span)
        return lower, upper, weight

    def _bilinear(field_2d, lat_target, lon_target):
        lat_target = float(np.clip(lat_target, lat_values[0], lat_values[-1]))
        lat_lower, lat_upper, lat_weight = _interp_axis(lat_values, lat_target)
        lon_lower, lon_upper, lon_weight = _interp_lon(lon_target)

        v00 = float(field_2d[lat_lower, lon_lower])
        v01 = float(field_2d[lat_lower, lon_upper])
        v10 = float(field_2d[lat_upper, lon_lower])
        v11 = float(field_2d[lat_upper, lon_upper])

        return (
            (1.0 - lat_weight) * ((1.0 - lon_weight) * v00 + lon_weight * v01)
            + lat_weight * ((1.0 - lon_weight) * v10 + lon_weight * v11)
        )

    def _sample_uv(t_target, lat_target, lon_target):
        time_target_ns = pd.Timestamp(t_target).value
        time_lower, time_upper, time_weight = _interp_axis(time_values_ns, time_target_ns)

        u_lower = _bilinear(u_values[time_lower], lat_target, lon_target)
        v_lower = _bilinear(v_values[time_lower], lat_target, lon_target)
        if time_lower == time_upper:
            return u_lower, v_lower

        u_upper = _bilinear(u_values[time_upper], lat_target, lon_target)
        v_upper = _bilinear(v_values[time_upper], lat_target, lon_target)
        u_interp = (1.0 - time_weight) * u_lower + time_weight * u_upper
        v_interp = (1.0 - time_weight) * v_lower + time_weight * v_upper
        return u_interp, v_interp

    lat = float(start_lat)
    lon = float(start_lon) % 360.0
    t_curr = t_nearest
    elapsed_hours = 0

    records = [
        {
            "step_hour": elapsed_hours,
            "valid_time": t_curr,
            "latitude": lat,
            "longitude": lon,
        }
    ]

    for step_hour in tqdm(
        step_hours,
        desc=f"Backward integration ({time_step_hours}h steps)",
        unit="step",
    ):
        dt_step_s = float(step_hour) * 3600.0
        dt_sub_s = dt_step_s / substeps
        lat_step = lat
        lon_step = lon

        for s in range(substeps):
            sec_back = (s + 0.5) * dt_sub_s
            t_mid = t_curr - pd.Timedelta(seconds=sec_back)
            u_ms, v_ms = _sample_uv(t_mid, lat_step, lon_step)

            dlat_deg = np.degrees((v_ms * dt_sub_s) / earth_radius_m)
            coslat = max(np.cos(np.radians(lat_step)), 1e-6)
            dlon_deg = np.degrees((u_ms * dt_sub_s) / (earth_radius_m * coslat))

            lat_step = float(np.clip(lat_step - dlat_deg, -89.75, 89.75))
            lon_step = float((lon_step - dlon_deg) % 360.0)

        t_curr = t_curr - pd.Timedelta(hours=step_hour)
        elapsed_hours += step_hour
        lat = lat_step
        lon = lon_step

        records.append(
            {
                "step_hour": elapsed_hours,
                "valid_time": t_curr,
                "latitude": lat,
                "longitude": lon,
            }
        )

    return pd.DataFrame(records)


trajectory_end_time_utc = pd.Timestamp(
    ds["valid_time"].sel(valid_time=TRAJECTORY_END_TIME_UTC.to_datetime64(), method="nearest").values
).round("h")
trajectory_hours_back = int(
    (trajectory_end_time_utc - TRAJECTORY_START_TIME_UTC).total_seconds() // 3600
)
if trajectory_hours_back < 0:
    raise RuntimeError("TRAJECTORY_START_TIME_UTC must be earlier than TRAJECTORY_END_TIME_UTC.")

trajectory_df = backward_integrate_trajectory_uv(
    ds,
    start_lat=VANCOUVER_LAT,
    start_lon=VANCOUVER_LON,
    start_time=trajectory_end_time_utc,
    pressure_level=925,
    hours_back=trajectory_hours_back,
    time_step_hours=TRAJECTORY_TIME_STEP_HOURS,
    substeps=4,
)

trajectory_start_time_utc = pd.Timestamp(trajectory_df["valid_time"].min()).round("h")
TRAJECTORY_POINT_LABEL = "Trajectory sampled points"
TRAJECTORY_TITLE_LABEL = f"{TRAJECTORY_TIME_STEP_HOURS}h sampled backward trajectory"
TRAJECTORY_LEGEND_LABEL = (
    f"Backward trajectory ({trajectory_start_time_utc.strftime('%Y-%m-%d %H:%M')} to "
    f"{trajectory_end_time_utc.strftime('%Y-%m-%d %H:%M')} UTC)"
)

trajectory_df.head()


In [ ]:
# Gradient/extrema helper functions copied from causes-of-gph-extremas.ipynb.
GRAD_STEP_KM = 35.0
GRAD_PROBE_DEG = 0.20
GRAD_MIN_MAG_M_PER_KM = 0.02
GRAD_MAX_STEPS = 220
GRAD_MONOTONIC_TOL_M = 1e-3


def _interp_gph_value_local(gph2d, lon, lat):
    val = gph2d.interp(
        latitude=float(lat),
        longitude=float(lon),
        kwargs={"fill_value": "extrapolate"},
    )
    return float(val.values)


def _km_per_deg_lon(lat):
    return max(111.32 * np.cos(np.deg2rad(float(lat))), 1e-6)


def _local_gph_gradient_east_north(gph2d, lon, lat, probe_deg=0.20):
    lon = float(lon)
    lat = float(lat)
    h = float(probe_deg)

    g_lon_plus = _interp_gph_value_local(gph2d, lon + h, lat)
    g_lon_minus = _interp_gph_value_local(gph2d, lon - h, lat)
    g_lat_plus = _interp_gph_value_local(gph2d, lon, lat + h)
    g_lat_minus = _interp_gph_value_local(gph2d, lon, lat - h)

    dgd_lon_deg = (g_lon_plus - g_lon_minus) / (2.0 * h)
    dgd_lat_deg = (g_lat_plus - g_lat_minus) / (2.0 * h)

    dgd_east = dgd_lon_deg / _km_per_deg_lon(lat)
    dgd_north = dgd_lat_deg / 111.32

    return np.array([float(dgd_east), float(dgd_north)], dtype=float)


def _trace_gradient_path(
    gph2d,
    lon0,
    lat0,
    prefer="increase",
    step_km=35.0,
    probe_deg=0.20,
    grad_min_mag=0.02,
    max_steps=220,
    monotonic_tol=1e-3,
):
    lon0 = float(lon0)
    lat0 = float(lat0)

    base_val = _interp_gph_value_local(gph2d, lon0, lat0)
    samples = [(lon0, lat0, base_val)]
    stop_reason = "max_steps"

    for _ in range(max_steps):
        curr_lon, curr_lat, curr_val = samples[-1]

        grad = _local_gph_gradient_east_north(
            gph2d,
            curr_lon,
            curr_lat,
            probe_deg=probe_deg,
        )
        grad_mag = float(np.linalg.norm(grad))

        if not np.isfinite(grad_mag) or grad_mag < float(grad_min_mag):
            stop_reason = "gradient_too_small"
            break

        step_dir = grad / grad_mag
        if prefer == "decrease":
            step_dir = -step_dir

        d_east_km = float(step_dir[0]) * float(step_km)
        d_north_km = float(step_dir[1]) * float(step_km)

        next_lon = curr_lon + d_east_km / _km_per_deg_lon(curr_lat)
        next_lat = curr_lat + d_north_km / 111.32

        if (
            next_lon < region_lon_min
            or next_lon > region_lon_max
            or next_lat < region_lat_min
            or next_lat > region_lat_max
        ):
            stop_reason = "domain_edge"
            break

        next_val = _interp_gph_value_local(gph2d, next_lon, next_lat)
        if not np.isfinite(next_val):
            stop_reason = "nan"
            break

        if prefer == "decrease":
            if not (next_val < curr_val - monotonic_tol):
                stop_reason = "cannot_decrease"
                break
        else:
            if not (next_val > curr_val + monotonic_tol):
                stop_reason = "cannot_increase"
                break

        samples.append((float(next_lon), float(next_lat), float(next_val)))

    line = np.asarray([(p[0], p[1]) for p in samples], dtype=float)
    final_lon, final_lat, final_val = samples[-1]

    return {
        "line": line,
        "final_lon": float(final_lon),
        "final_lat": float(final_lat),
        "final_value": float(final_val),
        "stop_reason": stop_reason,
        "num_steps": int(len(samples) - 1),
    }


def _nearest_contour_segment_to_point(gph2d, levels, lon_ref, lat_ref):
    levels = np.asarray(levels, dtype=float)
    levels = np.unique(np.sort(levels[np.isfinite(levels)]))
    if levels.size == 0:
        return None

    fig, ax = plt.subplots(figsize=(3, 2))
    cs = ax.contour(
        gph2d["longitude"].values,
        gph2d["latitude"].values,
        gph2d.values,
        levels=levels,
    )
    plt.close(fig)

    lon_ref = float(lon_ref)
    lat_ref = float(lat_ref)
    lon_scale = max(np.cos(np.deg2rad(lat_ref)), 1e-6)

    best = None

    for level, segs in zip(cs.levels, cs.allsegs):
        for seg in segs:
            seg = np.asarray(seg, dtype=float)
            if seg.shape[0] < 2:
                continue

            d = np.sqrt(((seg[:, 0] - lon_ref) * lon_scale) ** 2 + (seg[:, 1] - lat_ref) ** 2)
            idx = int(np.argmin(d))
            dist = float(d[idx])

            if best is None or dist < best["distance"]:
                best = {
                    "level": float(level),
                    "segment": seg,
                    "nearest_point": seg[idx],
                    "distance": dist,
                }

    return best


In [ ]:
# Reusable precompute: GPH extrema for sampled trajectory points.
FRAME_STEP_HOURS = 6


def _lat_slice_from_bounds(lat_coord, lat_min, lat_max):
    lat_values = np.asarray(lat_coord.values, dtype=float)
    if lat_values[0] > lat_values[-1]:
        return slice(float(lat_max), float(lat_min))
    return slice(float(lat_min), float(lat_max))


def _build_trajectory_frame_rows(trajectory_df, frame_step_hours=6):
    traj = trajectory_df.copy()
    traj["valid_time"] = pd.to_datetime(traj["valid_time"])
    traj["longitude_360"] = traj["longitude"] % 360.0

    if "step_hour" in traj.columns:
        frame_rows = (
            traj.loc[traj["step_hour"] % frame_step_hours == 0]
            .sort_values("step_hour", ascending=False)
            .reset_index(drop=True)
        )
        trajectory_line = traj.sort_values("step_hour", ascending=False).copy()
    else:
        trajectory_line = traj.sort_values("valid_time", ascending=True).copy()
        frame_rows = trajectory_line.iloc[::frame_step_hours].copy().reset_index(drop=True)
        frame_rows["step_hour"] = np.nan

    if frame_rows.empty:
        raise RuntimeError(f"No {frame_step_hours}-hour trajectory points found.")

    frame_times = pd.to_datetime(frame_rows["valid_time"]).dt.round("h")
    frame_times_da = xr.DataArray(frame_times.to_numpy(dtype="datetime64[ns]"), dims="frame")

    return {
        "trajectory_df": traj,
        "trajectory_line": trajectory_line,
        "frame_rows": frame_rows,
        "frame_times": frame_times,
        "frame_times_da": frame_times_da,
    }


def compute_gph_extrema_for_trajectory_points(
    trajectory_df,
    gph_925_m_all,
    frame_step_hours=6,
    extent_lon_min=None,
    extent_lon_max=None,
    extent_lat_min=23.0,
    extent_lat_max=None,
    extrema_search_levels=None,
):
    extent_lon_min = float(GPH_REGION_LON_MIN if extent_lon_min is None else extent_lon_min)
    extent_lon_max = float(GPH_REGION_LON_MAX if extent_lon_max is None else extent_lon_max)
    extent_lat_max = float(GPH_REGION_LAT_MAX if extent_lat_max is None else extent_lat_max)
    extent_lat_min = float(extent_lat_min)

    global region_lon_min, region_lon_max, region_lat_min, region_lat_max
    region_lon_min = extent_lon_min
    region_lon_max = extent_lon_max
    region_lat_min = extent_lat_min
    region_lat_max = extent_lat_max

    traj_context = _build_trajectory_frame_rows(trajectory_df, frame_step_hours=frame_step_hours)
    frame_rows = traj_context["frame_rows"]
    frame_times_da = traj_context["frame_times_da"]

    lat_slice = _lat_slice_from_bounds(gph_925_m_all["latitude"], extent_lat_min, extent_lat_max)
    gph_region = gph_925_m_all.sel(
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    )
    gph_frames = gph_region.sel(valid_time=frame_times_da, method="nearest").load()

    if extrema_search_levels is None:
        extrema_search_levels = np.asarray(
            globals().get("gph_contour_levels", np.arange(560.0, 941.0, 20.0)),
            dtype=float,
        )
    extrema_search_levels = np.unique(np.sort(extrema_search_levels[np.isfinite(extrema_search_levels)]))

    records = []
    summary_rows = []

    for i, row in tqdm(frame_rows.iterrows(), total=len(frame_rows), desc="Tracing extrema", unit="frame"):
        frame_ts = pd.Timestamp(row["valid_time"]).round("h")
        frame_step = row.get("step_hour", np.nan)
        gph_frame = gph_frames.isel(frame=i)

        lon_curr = float(row["longitude_360"])
        lat_curr = float(row["latitude"])

        dec_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="decrease",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )
        inc_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="increase",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )

        dec_closest = _nearest_contour_segment_to_point(
            gph_frame,
            extrema_search_levels,
            dec_trace["final_lon"],
            dec_trace["final_lat"],
        )
        inc_closest = _nearest_contour_segment_to_point(
            gph_frame,
            extrema_search_levels,
            inc_trace["final_lon"],
            inc_trace["final_lat"],
        )

        long_end_a = np.array([dec_trace["final_lon"], dec_trace["final_lat"]], dtype=float)
        long_end_b = np.array([inc_trace["final_lon"], inc_trace["final_lat"]], dtype=float)

        extrema_levels = []
        if dec_closest is not None:
            extrema_levels.append(float(dec_closest["level"]))
        if inc_closest is not None:
            extrema_levels.append(float(inc_closest["level"]))
        extrema_levels = np.unique(np.asarray(extrema_levels, dtype=float))

        record = {
            "frame_index": int(i),
            "frame_time": frame_ts,
            "step_hour": frame_step,
            "lon_curr": lon_curr,
            "lat_curr": lat_curr,
            "gph_frame": gph_frame,
            "dec_trace": dec_trace,
            "inc_trace": inc_trace,
            "dec_closest": dec_closest,
            "inc_closest": inc_closest,
            "long_end_a": long_end_a,
            "long_end_b": long_end_b,
            "extrema_levels": extrema_levels,
        }
        records.append(record)

        summary_rows.append(
            {
                "frame_index": int(i),
                "valid_time": frame_ts,
                "step_hour": frame_step,
                "curr_lon": lon_curr,
                "curr_lat": lat_curr,
                "dec_level_m": float(dec_closest["level"]) if dec_closest is not None else np.nan,
                "inc_level_m": float(inc_closest["level"]) if inc_closest is not None else np.nan,
                "dec_stop_reason": dec_trace["stop_reason"],
                "inc_stop_reason": inc_trace["stop_reason"],
            }
        )

    return {
        **traj_context,
        "gph_frames": gph_frames,
        "records": records,
        "extrema_df": pd.DataFrame(summary_rows),
        "extrema_search_levels": extrema_search_levels,
        "lat_slice": lat_slice,
        "extent_lon_min": extent_lon_min,
        "extent_lon_max": extent_lon_max,
        "extent_lat_min": extent_lat_min,
        "extent_lat_max": extent_lat_max,
        "frame_step_hours": frame_step_hours,
    }


extrema_context = compute_gph_extrema_for_trajectory_points(
    trajectory_df=trajectory_df,
    gph_925_m_all=gph_925_m_all,
    frame_step_hours=FRAME_STEP_HOURS,
)

extrema_context["extrema_df"].head()


In [ ]:
# Precompute 925 hPa thermodynamic backgrounds at the same frame times.

def _time_coord_name(ds):
    if "valid_time" in ds.coords:
        return "valid_time"
    if "time" in ds.coords:
        return "time"
    raise RuntimeError("Expected a valid_time/time coordinate in dataset.")


def compute_925hpa_virtual_temperature_and_thetae(extrema_context, pressure_level_hpa=925.0):
    temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
    humid_ds = open_dataset_from_data_dir(SPECIFIC_HUMIDITY_FILE)

    temp_time_coord = _time_coord_name(temp_ds)
    humid_time_coord = _time_coord_name(humid_ds)

    if "t" not in temp_ds.data_vars:
        raise RuntimeError("Expected variable 't' in temperature dataset.")
    if "q" not in humid_ds.data_vars:
        raise RuntimeError("Expected variable 'q' in specific humidity dataset.")
    if "pressure_level" not in temp_ds.coords or "pressure_level" not in humid_ds.coords:
        raise RuntimeError("Expected pressure_level coordinate in both temperature and humidity datasets.")

    frame_times_da = extrema_context["frame_times_da"]
    lat_slice = extrema_context["lat_slice"]
    extent_lon_min = float(extrema_context["extent_lon_min"])
    extent_lon_max = float(extrema_context["extent_lon_max"])

    t_region = temp_ds["t"].sel(
        pressure_level=pressure_level_hpa,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    )
    q_region = humid_ds["q"].sel(
        pressure_level=pressure_level_hpa,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    )

    t_frames = t_region.sel({temp_time_coord: frame_times_da}, method="nearest").load()
    q_frames = q_region.sel({humid_time_coord: frame_times_da}, method="nearest").load()
    q_frames = xr.apply_ufunc(np.clip, q_frames, 1e-8, 0.5)

    mixing_ratio = mpcalc.mixing_ratio_from_specific_humidity(
        q_frames.values * units.dimensionless
    )
    virtual_temp = mpcalc.virtual_temperature(
        t_frames.values * units.kelvin,
        mixing_ratio,
    ).magnitude

    dewpoint = mpcalc.dewpoint_from_specific_humidity(
        pressure_level_hpa * units.hPa,
        t_frames.values * units.kelvin,
        q_frames.values * units.dimensionless,
    )
    theta_e = mpcalc.equivalent_potential_temperature(
        pressure_level_hpa * units.hPa,
        t_frames.values * units.kelvin,
        dewpoint,
    ).magnitude

    virtual_temp_frames = xr.DataArray(
        virtual_temp,
        coords=t_frames.coords,
        dims=t_frames.dims,
        name=f"virtual_temperature_{int(np.round(pressure_level_hpa))}hpa",
        attrs={"units": "K"},
    )
    theta_e_frames = xr.DataArray(
        theta_e,
        coords=t_frames.coords,
        dims=t_frames.dims,
        name=f"theta_e_{int(np.round(pressure_level_hpa))}hpa",
        attrs={"units": "K"},
    )

    temp_ds.close()
    humid_ds.close()

    return {
        "virtual_temp_frames": virtual_temp_frames,
        "theta_e_frames": theta_e_frames,
        "pressure_level_hpa": float(pressure_level_hpa),
    }


thermo_context = compute_925hpa_virtual_temperature_and_thetae(
    extrema_context,
    pressure_level_hpa=925.0,
)
virtual_temp_frames = thermo_context["virtual_temp_frames"]
theta_e_frames = thermo_context["theta_e_frames"]

virtual_temp_frames


In [ ]:
# Compute 925 hPa virtual temperature and theta-e at each sampled trajectory point.

def compute_trajectory_thermo_series(trajectory_df, pressure_level_hpa=925.0):
    temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
    humid_ds = open_dataset_from_data_dir(SPECIFIC_HUMIDITY_FILE)

    try:
        temp_time_coord = _time_coord_name(temp_ds)
        humid_time_coord = _time_coord_name(humid_ds)

        traj = trajectory_df.copy()
        traj["valid_time"] = pd.to_datetime(traj["valid_time"])
        traj["longitude_360"] = traj["longitude"] % 360.0

        point_times = xr.DataArray(traj["valid_time"].to_numpy(dtype="datetime64[ns]"), dims="point")
        point_lats = xr.DataArray(traj["latitude"].to_numpy(dtype=float), dims="point")
        point_lons = xr.DataArray(traj["longitude_360"].to_numpy(dtype=float), dims="point")

        t_925 = temp_ds["t"].sel(pressure_level=pressure_level_hpa)
        q_925 = humid_ds["q"].sel(pressure_level=pressure_level_hpa)

        # Use nearest time match first, then spatial interpolation. For vectorized
        # multidimensional interpolation, SciPy expects fill_value=None (not "extrapolate").
        t_at_time = t_925.sel({temp_time_coord: point_times}, method="nearest")
        q_at_time = q_925.sel({humid_time_coord: point_times}, method="nearest")

        interp_kwargs = {"bounds_error": False, "fill_value": None}
        t_points = t_at_time.interp(
            latitude=point_lats,
            longitude=point_lons,
            kwargs=interp_kwargs,
        ).load()
        q_points = q_at_time.interp(
            latitude=point_lats,
            longitude=point_lons,
            kwargs=interp_kwargs,
        ).load()

        q_points = xr.apply_ufunc(np.clip, q_points, 1e-8, 0.5)

        mixing_ratio = mpcalc.mixing_ratio_from_specific_humidity(
            q_points.values * units.dimensionless
        )
        virtual_temp = mpcalc.virtual_temperature(
            t_points.values * units.kelvin,
            mixing_ratio,
        ).magnitude

        dewpoint = mpcalc.dewpoint_from_specific_humidity(
            pressure_level_hpa * units.hPa,
            t_points.values * units.kelvin,
            q_points.values * units.dimensionless,
        )
        theta_e = mpcalc.equivalent_potential_temperature(
            pressure_level_hpa * units.hPa,
            t_points.values * units.kelvin,
            dewpoint,
        ).magnitude

        out = traj[["step_hour", "valid_time", "latitude", "longitude", "longitude_360"]].copy()
        out["virtual_temperature_K"] = np.asarray(virtual_temp, dtype=float)
        out["theta_e_K"] = np.asarray(theta_e, dtype=float)
        if "step_hour" in out.columns:
            out = out.sort_values("step_hour", ascending=True).reset_index(drop=True)
        else:
            out = out.sort_values("valid_time", ascending=True).reset_index(drop=True)
        return out
    finally:
        temp_ds.close()
        humid_ds.close()


trajectory_thermo_df = compute_trajectory_thermo_series(
    trajectory_df,
    pressure_level_hpa=925.0,
)

trajectory_thermo_df.head()


In [ ]:
# Line graph: 925 hPa virtual temperature at each sampled trajectory point.
plot_df = trajectory_thermo_df.sort_values("step_hour", ascending=True).copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    plot_df["step_hour"],
    plot_df["virtual_temperature_K"],
    color="tab:blue",
    linewidth=2.0,
    marker="o",
    markersize=3.5,
)
ax.set_xlabel("Backward trajectory step hour")
ax.set_ylabel("Virtual temperature (K)")
ax.set_title(f"925 hPa virtual temperature along the {TRAJECTORY_TITLE_LABEL}")
ax.grid(True, linestyle="--", alpha=0.45)
ax.set_xlim(float(plot_df["step_hour"].max()), float(plot_df["step_hour"].min()))
plt.tight_layout()
plt.show()


In [ ]:
# Line graph: 925 hPa theta-e at each sampled trajectory point.
plot_df = trajectory_thermo_df.sort_values("step_hour", ascending=True).copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    plot_df["step_hour"],
    plot_df["theta_e_K"],
    color="tab:orange",
    linewidth=2.0,
    marker="o",
    markersize=3.5,
)
ax.set_xlabel("Backward trajectory step hour")
ax.set_ylabel("Theta-e (K)")
ax.set_title(f"925 hPa equivalent potential temperature (theta-e) along the {TRAJECTORY_TITLE_LABEL}")
ax.grid(True, linestyle="--", alpha=0.45)
ax.set_xlim(float(plot_df["step_hour"].max()), float(plot_df["step_hour"].min()))
plt.tight_layout()
plt.show()


In [ ]:
# Plot helper that keeps the same trajectory/extrema structure as the old cell,
# while swapping the background field.

def _build_discrete_colormap(field_frames, bin_width, cmap_name):
    bin_width = float(bin_width)
    vmin = float(np.floor(float(field_frames.min().values) / bin_width) * bin_width)
    vmax = float(np.ceil(float(field_frames.max().values) / bin_width) * bin_width)

    if not np.isfinite(vmin) or not np.isfinite(vmax):
        raise RuntimeError("Non-finite background range encountered.")
    if vmax <= vmin:
        vmax = vmin + bin_width

    levels = np.arange(vmin, vmax + bin_width, bin_width)
    nbins = max(len(levels) - 1, 1)
    cmap = plt.get_cmap(cmap_name, nbins)
    norm = mcolors.BoundaryNorm(levels, ncolors=nbins, clip=True)

    return levels, cmap, norm


def plot_background_with_extrema(
    extrema_context,
    background_frames,
    field_name,
    colorbar_label,
    cmap_name="Spectral_r",
    bin_width=1.0,
):
    frame_rows = extrema_context["frame_rows"]
    trajectory_line = extrema_context["trajectory_line"]
    gph_frames = extrema_context["gph_frames"]
    records = extrema_context["records"]

    if "frame" not in background_frames.dims:
        raise RuntimeError("background_frames must have a 'frame' dimension.")
    if background_frames.sizes["frame"] != len(records):
        raise RuntimeError("background_frames frame count must match extrema records.")

    line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
    line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

    extent_lon_min = float(extrema_context["extent_lon_min"])
    extent_lon_max = float(extrema_context["extent_lon_max"])
    extent_lat_min = float(extrema_context["extent_lat_min"])
    extent_lat_max = float(extrema_context["extent_lat_max"])

    levels, cmap, norm = _build_discrete_colormap(background_frames, bin_width, cmap_name)

    for i, row in frame_rows.iterrows():
        record = records[i]
        frame_ts = pd.Timestamp(row["valid_time"]).round("h")
        frame_step = row.get("step_hour", np.nan)

        background_frame = background_frames.isel(frame=i)
        gph_frame = gph_frames.isel(frame=i)

        lon_curr = float(row["longitude_360"])
        lat_curr = float(row["latitude"])

        long_end_a = np.asarray(record["long_end_a"], dtype=float)
        long_end_b = np.asarray(record["long_end_b"], dtype=float)
        extrema_levels = np.asarray(record["extrema_levels"], dtype=float)

        fig = plt.figure(figsize=(14, 7))
        ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

        ax.coastlines(resolution="110m", linewidth=0.8, color="black")
        ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

        mesh = ax.pcolormesh(
            background_frame["longitude"],
            background_frame["latitude"],
            background_frame,
            cmap=cmap,
            norm=norm,
            transform=ccrs.PlateCarree(),
            shading="auto",
            zorder=1,
        )

        contour_handle_for_legend = None
        if extrema_levels.size > 0:
            ax.contour(
                gph_frame["longitude"],
                gph_frame["latitude"],
                gph_frame,
                levels=extrema_levels,
                colors="black",
                linewidths=1.7,
                alpha=0.95,
                transform=ccrs.PlateCarree(),
                zorder=7,
            )
            contour_handle_for_legend = Line2D([0], [0], color="black", lw=1.7)

        (between_extrema_handle,) = ax.plot(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            color="magenta",
            linewidth=2.0,
            linestyle="-",
            transform=ccrs.PlateCarree(),
            zorder=9,
            label="Line between extrema stops",
        )

        extrema_endpoints_handle = ax.scatter(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            s=42,
            c="magenta",
            marker="x",
            linewidths=1.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label="Extrema stops",
        )

        (traj_line_handle,) = ax.plot(
            line_lons,
            line_lats,
            color="white",
            linewidth=1.8,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label="Backward trajectory (72h to 0h)",
        )

        traj_points_handle = ax.scatter(
            line_lons,
            line_lats,
            s=18,
            c="black",
            alpha=0.35,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label=TRAJECTORY_POINT_LABEL,
        )

        current_handle = ax.scatter(
            [lon_curr],
            [lat_curr],
            marker="x",
            s=130,
            c="deepskyblue",
            linewidths=2.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label=f"Current {extrema_context['frame_step_hours']}h point",
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False

        step_text = (
            f"t-{int(frame_step)}h"
            if pd.notna(frame_step)
            else f"{extrema_context['frame_step_hours']}h sample"
        )
        ax.set_title(
            f"{field_name} + extrema contours | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
        )

        cbar = plt.colorbar(
            mesh,
            ax=ax,
            boundaries=levels,
            orientation="horizontal",
            pad=0.05,
            shrink=0.9,
        )
        cbar.set_label(colorbar_label)

        legend_handles = [
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ]
        legend_labels = [
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {extrema_context['frame_step_hours']}h point",
            "Line between extrema stops",
            "Extrema stops",
        ]
        if contour_handle_for_legend is not None:
            legend_handles.append(contour_handle_for_legend)
            legend_labels.append("Full contour lines at extrema levels")

        ax.legend(legend_handles, legend_labels, loc="lower left")
        plt.tight_layout()
        plt.show()


In [ ]:
# Same style as old cell, but background = 925 hPa virtual temperature.
plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=virtual_temp_frames,
    field_name="925 hPa virtual temperature",
    colorbar_label="925 hPa virtual temperature (K)",
    cmap_name="coolwarm",
    bin_width=1.0,
)


In [ ]:
# Same style as old cell, but background = 925 hPa equivalent potential temperature.
plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=theta_e_frames,
    field_name="925 hPa equivalent potential temperature (theta-e)",
    colorbar_label="925 hPa theta-e (K)",
    cmap_name="viridis",
    bin_width=2.0,
)


In [ ]:
# Same style as old cell, but background = 925 hPa virtual temperature gradient magnitude.
lat_rad = np.deg2rad(virtual_temp_frames["latitude"])
km_per_deg_lon = 111.32 * xr.DataArray(
    np.clip(np.cos(lat_rad), 1e-6, None),
    coords={"latitude": virtual_temp_frames["latitude"]},
    dims=("latitude",),
)

# Differentiate in lat/lon space, then convert to east/north gradients in K/km.
dvt_dlat_deg = virtual_temp_frames.differentiate("latitude")
dvt_dlon_deg = virtual_temp_frames.differentiate("longitude")
dvt_dnorth_km = dvt_dlat_deg / 111.32
dvt_deast_km = dvt_dlon_deg / km_per_deg_lon

virtual_temp_grad_mag_100km = xr.apply_ufunc(np.hypot, dvt_deast_km, dvt_dnorth_km) * 100.0
virtual_temp_grad_mag_100km = virtual_temp_grad_mag_100km.rename(
    "virtual_temperature_gradient_925hpa_100km"
)
virtual_temp_grad_mag_100km.attrs["units"] = "K (100 km)^-1"
virtual_temp_grad_mag_100km = virtual_temp_grad_mag_100km.clip(min=0.0, max=10.0)

plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=virtual_temp_grad_mag_100km,
    field_name="925 hPa virtual temperature gradient magnitude",
    colorbar_label="|∇Tv| (K per 100 km)",
    cmap_name="magma",
    bin_width=0.2,
)


In [ ]:
# Same style as old cell, but background = 925 hPa wind speed.
wind_time_coord = _time_coord_name(gph_source_dataset)
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

u_frames = gph_source_dataset["u"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

v_frames = gph_source_dataset["v"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

wind_speed_frames = xr.apply_ufunc(np.hypot, u_frames, v_frames).rename("wind_speed_925hpa")
wind_speed_frames.attrs["units"] = "m s^-1"

plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=wind_speed_frames,
    field_name="925 hPa wind speed",
    colorbar_label="925 hPa wind speed (m s^-1)",
    cmap_name="YlGnBu",
    bin_width=2.0,
)


In [ ]:
# Same style as old cell, but background = 925 hPa frontogenesis.
wind_time_coord = _time_coord_name(gph_source_dataset)
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
try:
    temp_time_coord = _time_coord_name(temp_ds)

    t_frames = temp_ds["t"].sel(
        pressure_level=925.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({temp_time_coord: frame_times_da}, method="nearest").load()

    u_frames = gph_source_dataset["u"].sel(
        pressure_level=925.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({wind_time_coord: frame_times_da}, method="nearest").load()

    v_frames = gph_source_dataset["v"].sel(
        pressure_level=925.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({wind_time_coord: frame_times_da}, method="nearest").load()

    theta_frames = xr.DataArray(
        mpcalc.potential_temperature(
            925.0 * units.hPa,
            t_frames.values * units.kelvin,
        ).magnitude,
        coords=t_frames.coords,
        dims=t_frames.dims,
        name="theta_925hpa",
    )

    lat_rad = np.deg2rad(theta_frames["latitude"])
    meters_per_deg_lon = 111_320.0 * xr.DataArray(
        np.clip(np.cos(lat_rad), 1e-6, None),
        coords={"latitude": theta_frames["latitude"]},
        dims=("latitude",),
    )
    meters_per_deg_lat = 111_320.0

    dtheta_dx = theta_frames.differentiate("longitude") / meters_per_deg_lon
    dtheta_dy = theta_frames.differentiate("latitude") / meters_per_deg_lat

    du_dx = u_frames.differentiate("longitude") / meters_per_deg_lon
    du_dy = u_frames.differentiate("latitude") / meters_per_deg_lat
    dv_dx = v_frames.differentiate("longitude") / meters_per_deg_lon
    dv_dy = v_frames.differentiate("latitude") / meters_per_deg_lat

    grad_theta_mag = xr.apply_ufunc(np.hypot, dtheta_dx, dtheta_dy)
    grad_theta_mag_safe = xr.where(grad_theta_mag > 1e-12, grad_theta_mag, np.nan)

    numerator = (
        (dtheta_dx ** 2) * du_dx
        + (dtheta_dy ** 2) * dv_dy
        + (dtheta_dx * dtheta_dy) * (du_dy + dv_dx)
    )
    frontogenesis_si = -numerator / grad_theta_mag_safe

    # Convert from K m^-1 s^-1 to K (100 km)^-1 (3 h)^-1.
    frontogenesis_frames = frontogenesis_si * (100_000.0 * 3.0 * 3600.0)
    frontogenesis_frames = frontogenesis_frames.rename("frontogenesis_925hpa")
    frontogenesis_frames.attrs["units"] = "K (100 km)^-1 (3 h)^-1"
    frontogenesis_frames = frontogenesis_frames.clip(min=-5.0, max=5.0)
finally:
    temp_ds.close()

plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=frontogenesis_frames,
    field_name="925 hPa frontogenesis",
    colorbar_label="Frontogenesis (K per 100 km per 3 h)",
    cmap_name="RdBu_r",
    bin_width=0.5,
)


In [ ]:
# Same style as old cell, but background = 925 hPa temperature (discrete scale).
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
try:
    temp_time_coord = _time_coord_name(temp_ds)

    temp_925_frames = temp_ds["t"].sel(
        pressure_level=925.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({temp_time_coord: frame_times_da}, method="nearest").load()

    temp_925_c_frames = (temp_925_frames - 273.15).rename("temperature_925hpa_c")
    temp_925_c_frames.attrs["units"] = "degC"
finally:
    temp_ds.close()

plot_background_with_extrema(
    extrema_context=extrema_context,
    background_frames=temp_925_c_frames,
    field_name="925 hPa temperature (discrete)",
    colorbar_label="925 hPa temperature (degC)",
    cmap_name="coolwarm",
    bin_width=1.0,
)


In [ ]:
# Same style as old cell, but background = 925 hPa GPH gradient magnitude.
gph_frames_local = extrema_context["gph_frames"]
gph_grad_clip_max_100km = 30.0
gph_contour_spacing_m = 20.0
gph_grad_bin_width_100km = 2.0
spacing_label = f"{int(gph_contour_spacing_m)}" if float(gph_contour_spacing_m).is_integer() else f"{gph_contour_spacing_m:g}"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
records = extrema_context["records"]

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

lat_rad = np.deg2rad(gph_frames_local["latitude"])
km_per_deg_lon = 111.32 * xr.DataArray(
    np.clip(np.cos(lat_rad), 1e-6, None),
    coords={"latitude": gph_frames_local["latitude"]},
    dims=("latitude",),
)

# Horizontal gradient components in m/km.
dgph_dlat_deg = gph_frames_local.differentiate("latitude")
dgph_dlon_deg = gph_frames_local.differentiate("longitude")
dgph_dnorth_km = dgph_dlat_deg / 111.32
dgph_deast_km = dgph_dlon_deg / km_per_deg_lon

gph_grad_mag_100km = xr.apply_ufunc(np.hypot, dgph_deast_km, dgph_dnorth_km) * 100.0
gph_grad_mag_100km = gph_grad_mag_100km.clip(min=0.0, max=gph_grad_clip_max_100km)
gph_grad_mag_100km = gph_grad_mag_100km.rename("gph_gradient_925hpa_100km")
gph_grad_mag_100km.attrs["units"] = "m (100 km)^-1"

grad_levels, grad_cmap, grad_norm = _build_discrete_colormap(
    gph_grad_mag_100km,
    bin_width=gph_grad_bin_width_100km,
    cmap_name="cividis",
)

gph_vmin = float(np.floor(float(gph_frames_local.min().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames_local.max().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + gph_contour_spacing_m, gph_contour_spacing_m)

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)
    gph_frame = gph_frames_local.isel(frame=i)
    grad_frame = gph_grad_mag_100km.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])

    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)
    extrema_levels = np.asarray(record["extrema_levels"], dtype=float)

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        grad_frame["longitude"],
        grad_frame["latitude"],
        grad_frame,
        cmap=grad_cmap,
        norm=grad_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    contour_gph = ax.contour(
        gph_frame["longitude"],
        gph_frame["latitude"],
        gph_frame,
        levels=gph_contour_levels,
        colors="dimgray",
        linewidths=0.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    extrema_contour_handle = None
    if extrema_levels.size > 0:
        ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=extrema_levels,
            colors="black",
            linewidths=1.8,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        extrema_contour_handle = Line2D([0], [0], color="black", lw=1.8)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"925 hPa GPH gradient magnitude + 925 hPa GPH contours ({spacing_label} m spacing) | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=grad_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label(f"|∇GPH| (m per 100 km), clipped to {gph_grad_clip_max_100km:g}")

    contour_handle = Line2D([0], [0], color="dimgray", lw=0.9)
    legend_handles = [
        contour_handle,
        traj_line_handle,
        traj_points_handle,
        current_handle,
        between_extrema_handle,
        extrema_endpoints_handle,
    ]
    legend_labels = [
        f"925 hPa GPH contours ({spacing_label} m)",
        "Backward trajectory (72h to 0h)",
        TRAJECTORY_POINT_LABEL,
        f"Current {extrema_context['frame_step_hours']}h point",
        "Line between extrema stops",
        "Extrema stops",
    ]
    if extrema_contour_handle is not None:
        legend_handles.append(extrema_contour_handle)
        legend_labels.append("Full contour lines at extrema levels")

    ax.legend(legend_handles, legend_labels, loc="lower left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Same style as old cell, but 925 hPa GPH contours use configurable spacing.
gph_contour_spacing_m = 20.0
wind_speed_bin_width_ms = 2.0
spacing_label = f"{int(gph_contour_spacing_m)}" if float(gph_contour_spacing_m).is_integer() else f"{gph_contour_spacing_m:g}"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
gph_frames = extrema_context["gph_frames"]
records = extrema_context["records"]
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

wind_time_coord = _time_coord_name(gph_source_dataset)
u_frames = gph_source_dataset["u"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

v_frames = gph_source_dataset["v"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

wind_speed_frames = xr.apply_ufunc(np.hypot, u_frames, v_frames).rename("wind_speed_925hpa")
wind_speed_frames.attrs["units"] = "m s^-1"

wind_levels, wind_cmap, wind_norm = _build_discrete_colormap(
    wind_speed_frames,
    bin_width=wind_speed_bin_width_ms,
    cmap_name="YlGnBu",
)

gph_vmin = float(np.floor(float(gph_frames.min().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames.max().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + gph_contour_spacing_m, gph_contour_spacing_m)

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)
    gph_frame = gph_frames.isel(frame=i)
    wind_frame = wind_speed_frames.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])

    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)
    extrema_levels = np.asarray(record["extrema_levels"], dtype=float)

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        wind_frame["longitude"],
        wind_frame["latitude"],
        wind_frame,
        cmap=wind_cmap,
        norm=wind_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    contour_gph = ax.contour(
        gph_frame["longitude"],
        gph_frame["latitude"],
        gph_frame,
        levels=gph_contour_levels,
        colors="dimgray",
        linewidths=0.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    extrema_contour_handle = None
    if extrema_levels.size > 0:
        ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=extrema_levels,
            colors="black",
            linewidths=1.8,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        extrema_contour_handle = Line2D([0], [0], color="black", lw=1.8)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"925 hPa wind speed + GPH contours ({spacing_label} m spacing) | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=wind_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label("925 hPa wind speed (m s^-1)")

    contour_handle = Line2D([0], [0], color="dimgray", lw=0.9)
    legend_handles = [
        contour_handle,
        traj_line_handle,
        traj_points_handle,
        current_handle,
        between_extrema_handle,
        extrema_endpoints_handle,
    ]
    legend_labels = [
        f"925 hPa GPH contours ({spacing_label} m)",
        "Backward trajectory (72h to 0h)",
        TRAJECTORY_POINT_LABEL,
        f"Current {extrema_context['frame_step_hours']}h point",
        "Line between extrema stops",
        "Extrema stops",
    ]
    if extrema_contour_handle is not None:
        legend_handles.append(extrema_contour_handle)
        legend_labels.append("Full contour lines at extrema levels")

    ax.legend(legend_handles, legend_labels, loc="lower left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Same style as old cell, but background = 250 hPa temperature and contours = 925 hPa GPH.
gph_contour_spacing_m = 20.0
temp_bin_width_c = 2.0
spacing_label = f"{int(gph_contour_spacing_m)}" if float(gph_contour_spacing_m).is_integer() else f"{gph_contour_spacing_m:g}"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
gph_frames = extrema_context["gph_frames"]
records = extrema_context["records"]
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
try:
    temp_time_coord = _time_coord_name(temp_ds)
    temp_250_frames = temp_ds["t"].sel(
        pressure_level=250.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({temp_time_coord: frame_times_da}, method="nearest").load()

    temp_250_c_frames = (temp_250_frames - 273.15).rename("temperature_250hpa_c")
    temp_250_c_frames.attrs["units"] = "degC"
finally:
    temp_ds.close()

temp_levels, temp_cmap, temp_norm = _build_discrete_colormap(
    temp_250_c_frames,
    bin_width=temp_bin_width_c,
    cmap_name="coolwarm",
)

gph_vmin = float(np.floor(float(gph_frames.min().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames.max().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + gph_contour_spacing_m, gph_contour_spacing_m)

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)

    temp_frame = temp_250_c_frames.isel(frame=i)
    gph_frame = gph_frames.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])

    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)
    extrema_levels = np.asarray(record["extrema_levels"], dtype=float)

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        temp_frame["longitude"],
        temp_frame["latitude"],
        temp_frame,
        cmap=temp_cmap,
        norm=temp_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    contour_gph = ax.contour(
        gph_frame["longitude"],
        gph_frame["latitude"],
        gph_frame,
        levels=gph_contour_levels,
        colors="dimgray",
        linewidths=0.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    extrema_contour_handle = None
    if extrema_levels.size > 0:
        ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=extrema_levels,
            colors="black",
            linewidths=1.8,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        extrema_contour_handle = Line2D([0], [0], color="black", lw=1.8)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"250 hPa temperature + 925 hPa GPH contours ({spacing_label} m spacing) | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=temp_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label("250 hPa temperature (degC)")

    contour_handle = Line2D([0], [0], color="dimgray", lw=0.9)
    legend_handles = [
        contour_handle,
        traj_line_handle,
        traj_points_handle,
        current_handle,
        between_extrema_handle,
        extrema_endpoints_handle,
    ]
    legend_labels = [
        f"925 hPa GPH contours ({spacing_label} m)",
        "Backward trajectory (72h to 0h)",
        TRAJECTORY_POINT_LABEL,
        f"Current {extrema_context['frame_step_hours']}h point",
        "Line between extrema stops",
        "Extrema stops",
    ]
    if extrema_contour_handle is not None:
        legend_handles.append(extrema_contour_handle)
        legend_labels.append("Full contour lines at extrema levels")

    ax.legend(legend_handles, legend_labels, loc="lower left")
    plt.tight_layout()
    plt.show()


In [ ]:
# 925 hPa wind speed with 20 m GPH contours, highlighting the 620 m line in red.
gph_contour_spacing_m = 20.0
highlight_gph_level_m = 620.0
wind_speed_bin_width_ms = 2.0
spacing_label = f"{int(gph_contour_spacing_m)}" if float(gph_contour_spacing_m).is_integer() else f"{gph_contour_spacing_m:g}"
highlight_label = f"{int(highlight_gph_level_m)}" if float(highlight_gph_level_m).is_integer() else f"{highlight_gph_level_m:g}"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
gph_frames = extrema_context["gph_frames"]
records = extrema_context["records"]
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

wind_time_coord = _time_coord_name(gph_source_dataset)
u_frames = gph_source_dataset["u"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

v_frames = gph_source_dataset["v"].sel(
    pressure_level=925.0,
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
).sel({wind_time_coord: frame_times_da}, method="nearest").load()

wind_speed_frames = xr.apply_ufunc(np.hypot, u_frames, v_frames).rename("wind_speed_925hpa")
wind_speed_frames.attrs["units"] = "m s^-1"

wind_levels, wind_cmap, wind_norm = _build_discrete_colormap(
    wind_speed_frames,
    bin_width=wind_speed_bin_width_ms,
    cmap_name="YlGnBu",
)

gph_vmin = float(np.floor(float(gph_frames.min().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames.max().values) / gph_contour_spacing_m) * gph_contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + gph_contour_spacing_m, gph_contour_spacing_m)
base_gph_levels = gph_contour_levels[~np.isclose(gph_contour_levels, highlight_gph_level_m)]

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)
    gph_frame = gph_frames.isel(frame=i)
    wind_frame = wind_speed_frames.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])

    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)
    extrema_levels = np.asarray(record["extrema_levels"], dtype=float)
    extrema_levels = extrema_levels[~np.isclose(extrema_levels, highlight_gph_level_m)]

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        wind_frame["longitude"],
        wind_frame["latitude"],
        wind_frame,
        cmap=wind_cmap,
        norm=wind_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    if base_gph_levels.size > 0:
        contour_gph = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=base_gph_levels,
            colors="black",
            linewidths=0.8,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )
        ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    highlight_contour_handle = None
    gph_frame_min = float(gph_frame.min().values)
    gph_frame_max = float(gph_frame.max().values)
    if gph_frame_min <= highlight_gph_level_m <= gph_frame_max:
        contour_620 = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=[highlight_gph_level_m],
            colors="red",
            linewidths=1.8,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            zorder=8,
        )
        ax.clabel(contour_620, contour_620.levels, fmt="%.0f", inline=True, fontsize=8, colors="red")
        highlight_contour_handle = Line2D([0], [0], color="red", lw=1.8)

    extrema_contour_handle = None
    if extrema_levels.size > 0:
        ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=extrema_levels,
            colors="black",
            linewidths=1.8,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        extrema_contour_handle = Line2D([0], [0], color="black", lw=1.8)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"925 hPa wind speed + 925 hPa GPH contours ({spacing_label} m spacing, {highlight_label} m highlighted) | {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=wind_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label("925 hPa wind speed (m s^-1)")

    contour_handle = Line2D([0], [0], color="black", lw=0.9)
    legend_handles = [
        contour_handle,
        traj_line_handle,
        traj_points_handle,
        current_handle,
        between_extrema_handle,
        extrema_endpoints_handle,
    ]
    legend_labels = [
        f"925 hPa GPH contours ({spacing_label} m, black)",
        "Backward trajectory (72h to 0h)",
        TRAJECTORY_POINT_LABEL,
        f"Current {extrema_context['frame_step_hours']}h point",
        "Line between extrema stops",
        "Extrema stops",
    ]
    if highlight_contour_handle is not None:
        legend_handles.append(highlight_contour_handle)
        legend_labels.append(f"{highlight_label} m GPH contour (red)")
    if extrema_contour_handle is not None:
        legend_handles.append(extrema_contour_handle)
        legend_labels.append("Full contour lines at extrema levels")

    ax.legend(legend_handles, legend_labels, loc="lower left")
    plt.tight_layout()
    plt.show()


In [ ]:
# 250 hPa temperature background using the grey-contour / highlighted-contour helper.
temp_bin_width_c = 2.0

frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
try:
    temp_time_coord = _time_coord_name(temp_ds)
    temp_250_frames = temp_ds["t"].sel(
        pressure_level=250.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({temp_time_coord: frame_times_da}, method="nearest").load()

    temp_250_c_frames = (temp_250_frames - 273.15).rename("temperature_250hpa_c")
    temp_250_c_frames.attrs["units"] = "degC"
finally:
    temp_ds.close()

plot_background_with_gph_contour_highlight(
    extrema_context=extrema_context,
    background_frames=temp_250_c_frames,
    field_name="250 hPa temperature",
    colorbar_label="250 hPa temperature (degC)",
    cmap_name="coolwarm",
    bin_width=temp_bin_width_c,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
)


In [ ]:
# 925 hPa temperature background with all 20 m GPH contours and the 640 m contour highlighted.
temp_bin_width_c = 1.0

frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

temp_ds = open_dataset_from_data_dir(TEMPERATURE_FILE)
try:
    temp_time_coord = _time_coord_name(temp_ds)
    temp_925_frames = temp_ds["t"].sel(
        pressure_level=925.0,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({temp_time_coord: frame_times_da}, method="nearest").load()

    temp_925_c_frames = (temp_925_frames - 273.15).rename("temperature_925hpa_c")
    temp_925_c_frames.attrs["units"] = "degC"
finally:
    temp_ds.close()

plot_background_with_gph_contour_highlight(
    extrema_context=extrema_context,
    background_frames=temp_925_c_frames,
    field_name="925 hPa temperature",
    colorbar_label="925 hPa temperature (degC)",
    cmap_name="coolwarm",
    bin_width=temp_bin_width_c,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
)


In [ ]:
# Precompute 850/500/250 hPa GPH backgrounds for the grey-contour / highlighted-contour helper.
MULTI_LEVEL_GPH_FILE = "gph-250-500-850-1000-novdec2021.nc"

frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])

multi_level_gph_ds = open_dataset_from_data_dir(MULTI_LEVEL_GPH_FILE)
try:
    multi_level_gph_time_coord = _time_coord_name(multi_level_gph_ds)

    def _load_gph_frames_in_meters(pressure_level_hpa):
        z_frames = multi_level_gph_ds["z"].sel(
            pressure_level=pressure_level_hpa,
            longitude=slice(extent_lon_min, extent_lon_max),
            latitude=lat_slice,
        ).sel({multi_level_gph_time_coord: frame_times_da}, method="nearest").load()

        gph_frames = xr.apply_ufunc(np.divide, z_frames, 9.80665).rename(
            f"gph_{int(np.round(pressure_level_hpa))}hpa_m"
        )
        gph_frames.attrs["units"] = "m"
        return gph_frames

    gph_850_frames = _load_gph_frames_in_meters(850.0)
    gph_500_frames = _load_gph_frames_in_meters(500.0)
    gph_250_frames = _load_gph_frames_in_meters(250.0)
finally:
    multi_level_gph_ds.close()


In [ ]:
# 850 hPa GPH background with all 20 m 925 hPa GPH contours and the 640 m contour highlighted.
gph_bin_width_m = 30.0

plot_background_with_gph_contour_highlight(
    extrema_context=extrema_context,
    background_frames=gph_850_frames,
    field_name="850 hPa GPH",
    colorbar_label="850 hPa geopotential height (m)",
    cmap_name="viridis",
    bin_width=gph_bin_width_m,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
)


In [ ]:
# 500 hPa GPH background with all 20 m 925 hPa GPH contours and the 640 m contour highlighted.
gph_bin_width_m = 60.0

plot_background_with_gph_contour_highlight(
    extrema_context=extrema_context,
    background_frames=gph_500_frames,
    field_name="500 hPa GPH",
    colorbar_label="500 hPa geopotential height (m)",
    cmap_name="viridis",
    bin_width=gph_bin_width_m,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
)


In [ ]:
# 250 hPa GPH background with all 20 m 925 hPa GPH contours and the 640 m contour highlighted.
gph_bin_width_m = 120.0

plot_background_with_gph_contour_highlight(
    extrema_context=extrema_context,
    background_frames=gph_250_frames,
    field_name="250 hPa GPH",
    colorbar_label="250 hPa geopotential height (m)",
    cmap_name="viridis",
    bin_width=gph_bin_width_m,
    gph_contour_spacing_m=20.0,
    contour_color="dimgray",
    highlight_gph_level_m=640.0,
    highlight_color="green",
)


In [ ]:
# 500 hPa vertical velocity background (25x25 box mean) with 500 hPa GPH contours.
if "extrema_context" not in globals():
    raise RuntimeError("Run the extrema_context setup cell first so extrema_context exists.")
if "gph_500_frames" not in globals():
    raise RuntimeError("Run the 850/500/250 hPa GPH preload cell first so gph_500_frames exists.")
required_helpers = ["open_dataset_from_data_dir", "_time_coord_name"]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the helper/setup cells first so these functions exist: " + ", ".join(missing_helpers)
    )

import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

VERTICAL_VELOCITY_FILE = "era5_2021-nov_250-500-925_divergence_vertical_velocity.nc"
VERTICAL_VELOCITY_PRESSURE_LEVEL_HPA = 500.0
VERTICAL_VELOCITY_BIN_WIDTH_PA_S = 0.02
VERTICAL_VELOCITY_VMIN_PA_S = -2.0
VERTICAL_VELOCITY_VMAX_PA_S = 2.0
BOX_SIZE = 25
GPH_500_CONTOUR_SPACING_M = 20.0
GPH_500_CONTOUR_COLOR = "dimgray"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
records = extrema_context["records"]
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

if "frame" not in gph_500_frames.dims:
    raise RuntimeError("gph_500_frames must have a 'frame' dimension.")
if gph_500_frames.sizes["frame"] != len(records):
    raise RuntimeError("gph_500_frames frame count must match extrema records.")

box_size = int(BOX_SIZE)
if box_size < 1:
    raise RuntimeError("BOX_SIZE must be >= 1")
if box_size % 2 == 0:
    raise RuntimeError("BOX_SIZE must be odd so the box is centered on each pixel.")

vertical_velocity_ds = open_dataset_from_data_dir(VERTICAL_VELOCITY_FILE)
try:
    vv_time_coord = _time_coord_name(vertical_velocity_ds)
    if "pressure_level" not in vertical_velocity_ds.coords:
        raise RuntimeError("Expected pressure_level coordinate in vertical velocity dataset.")
    if "w" not in vertical_velocity_ds.data_vars:
        raise RuntimeError("Expected variable 'w' (vertical velocity) in dataset.")

    vv_frames = vertical_velocity_ds["w"].sel(
        pressure_level=VERTICAL_VELOCITY_PRESSURE_LEVEL_HPA,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({vv_time_coord: frame_times_da}, method="nearest").load()
finally:
    vertical_velocity_ds.close()

vv_frames_box = vv_frames.rolling(
    latitude=box_size,
    longitude=box_size,
    center=True,
    min_periods=1,
).mean().rename("vertical_velocity_500hpa_boxmean")
vv_frames_box.attrs["units"] = "Pa s^-1"

bin_width_pa_s = float(VERTICAL_VELOCITY_BIN_WIDTH_PA_S)
if not np.isfinite(bin_width_pa_s) or bin_width_pa_s <= 0.0:
    raise RuntimeError("VERTICAL_VELOCITY_BIN_WIDTH_PA_S must be a positive finite value.")
if VERTICAL_VELOCITY_VMAX_PA_S <= VERTICAL_VELOCITY_VMIN_PA_S:
    raise RuntimeError("Invalid vertical velocity plotting range.")

vv_levels = np.arange(
    VERTICAL_VELOCITY_VMIN_PA_S,
    VERTICAL_VELOCITY_VMAX_PA_S + bin_width_pa_s,
    bin_width_pa_s,
)
vv_nbins = max(len(vv_levels) - 1, 1)
vv_cmap = plt.get_cmap("RdBu", vv_nbins)
vv_norm = mcolors.BoundaryNorm(vv_levels, ncolors=vv_nbins, clip=True)

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

contour_spacing_label = (
    f"{int(GPH_500_CONTOUR_SPACING_M)}"
    if float(GPH_500_CONTOUR_SPACING_M).is_integer()
    else f"{GPH_500_CONTOUR_SPACING_M:g}"
)
gph_500_vmin = float(
    np.floor(float(gph_500_frames.min().values) / GPH_500_CONTOUR_SPACING_M) * GPH_500_CONTOUR_SPACING_M
)
gph_500_vmax = float(
    np.ceil(float(gph_500_frames.max().values) / GPH_500_CONTOUR_SPACING_M) * GPH_500_CONTOUR_SPACING_M
)
gph_500_contour_levels = np.arange(
    gph_500_vmin,
    gph_500_vmax + GPH_500_CONTOUR_SPACING_M,
    GPH_500_CONTOUR_SPACING_M,
)

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)

    vv_frame = vv_frames_box.isel(frame=i)
    gph_frame = gph_500_frames.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])
    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        vv_frame["longitude"],
        vv_frame["latitude"],
        vv_frame,
        cmap=vv_cmap,
        norm=vv_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    contour_500 = ax.contour(
        gph_frame["longitude"],
        gph_frame["latitude"],
        gph_frame,
        levels=gph_500_contour_levels,
        colors=GPH_500_CONTOUR_COLOR,
        linewidths=0.9,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    if len(contour_500.levels) > 0:
        ax.clabel(contour_500, contour_500.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"500 hPa vertical velocity (box-mean {box_size}x{box_size}, discrete, symmetric) + "
        f"500 hPa GPH contours ({contour_spacing_label} m spacing) | "
        f"{frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=vv_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label(
        "Vertical velocity (Pa s$^{-1}$): negative=upward motion (ascent), "
        "positive=downward motion (subsidence)"
    )

    contour_handle = Line2D([0], [0], color=GPH_500_CONTOUR_COLOR, lw=0.9)
    ax.legend(
        [
            contour_handle,
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ],
        [
            f"500 hPa GPH contours ({contour_spacing_label} m)",
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {extrema_context['frame_step_hours']}h point",
            "Line between extrema stops",
            "Extrema stops",
        ],
        loc="lower left",
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# 250 hPa vertical velocity background (25x25 box mean) with 250 hPa GPH contours.
if "extrema_context" not in globals():
    raise RuntimeError("Run the extrema_context setup cell first so extrema_context exists.")
if "gph_250_frames" not in globals():
    raise RuntimeError("Run the 850/500/250 hPa GPH preload cell first so gph_250_frames exists.")
required_helpers = ["open_dataset_from_data_dir", "_time_coord_name"]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the helper/setup cells first so these functions exist: " + ", ".join(missing_helpers)
    )

import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

VERTICAL_VELOCITY_FILE = "era5_2021-nov_250-500-925_divergence_vertical_velocity.nc"
VERTICAL_VELOCITY_PRESSURE_LEVEL_HPA = 250.0
VERTICAL_VELOCITY_BIN_WIDTH_PA_S = 0.02
VERTICAL_VELOCITY_VMIN_PA_S = -2.0
VERTICAL_VELOCITY_VMAX_PA_S = 2.0
BOX_SIZE = 25
GPH_250_CONTOUR_SPACING_M = 20.0
GPH_250_CONTOUR_COLOR = "dimgray"

frame_rows = extrema_context["frame_rows"]
trajectory_line = extrema_context["trajectory_line"]
records = extrema_context["records"]
frame_times_da = extrema_context["frame_times_da"]
lat_slice = extrema_context["lat_slice"]
extent_lon_min = float(extrema_context["extent_lon_min"])
extent_lon_max = float(extrema_context["extent_lon_max"])
extent_lat_min = float(extrema_context["extent_lat_min"])
extent_lat_max = float(extrema_context["extent_lat_max"])

if "frame" not in gph_250_frames.dims:
    raise RuntimeError("gph_250_frames must have a 'frame' dimension.")
if gph_250_frames.sizes["frame"] != len(records):
    raise RuntimeError("gph_250_frames frame count must match extrema records.")

box_size = int(BOX_SIZE)
if box_size < 1:
    raise RuntimeError("BOX_SIZE must be >= 1")
if box_size % 2 == 0:
    raise RuntimeError("BOX_SIZE must be odd so the box is centered on each pixel.")

vertical_velocity_ds = open_dataset_from_data_dir(VERTICAL_VELOCITY_FILE)
try:
    vv_time_coord = _time_coord_name(vertical_velocity_ds)
    if "pressure_level" not in vertical_velocity_ds.coords:
        raise RuntimeError("Expected pressure_level coordinate in vertical velocity dataset.")
    if "w" not in vertical_velocity_ds.data_vars:
        raise RuntimeError("Expected variable 'w' (vertical velocity) in dataset.")

    vv_frames = vertical_velocity_ds["w"].sel(
        pressure_level=VERTICAL_VELOCITY_PRESSURE_LEVEL_HPA,
        longitude=slice(extent_lon_min, extent_lon_max),
        latitude=lat_slice,
    ).sel({vv_time_coord: frame_times_da}, method="nearest").load()
finally:
    vertical_velocity_ds.close()

vv_frames_box = vv_frames.rolling(
    latitude=box_size,
    longitude=box_size,
    center=True,
    min_periods=1,
).mean().rename("vertical_velocity_250hpa_boxmean")
vv_frames_box.attrs["units"] = "Pa s^-1"

bin_width_pa_s = float(VERTICAL_VELOCITY_BIN_WIDTH_PA_S)
if not np.isfinite(bin_width_pa_s) or bin_width_pa_s <= 0.0:
    raise RuntimeError("VERTICAL_VELOCITY_BIN_WIDTH_PA_S must be a positive finite value.")
if VERTICAL_VELOCITY_VMAX_PA_S <= VERTICAL_VELOCITY_VMIN_PA_S:
    raise RuntimeError("Invalid vertical velocity plotting range.")

vv_levels = np.arange(
    VERTICAL_VELOCITY_VMIN_PA_S,
    VERTICAL_VELOCITY_VMAX_PA_S + bin_width_pa_s,
    bin_width_pa_s,
)
vv_nbins = max(len(vv_levels) - 1, 1)
vv_cmap = plt.get_cmap("RdBu", vv_nbins)
vv_norm = mcolors.BoundaryNorm(vv_levels, ncolors=vv_nbins, clip=True)

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

contour_spacing_label = (
    f"{int(GPH_250_CONTOUR_SPACING_M)}"
    if float(GPH_250_CONTOUR_SPACING_M).is_integer()
    else f"{GPH_250_CONTOUR_SPACING_M:g}"
)
gph_250_vmin = float(
    np.floor(float(gph_250_frames.min().values) / GPH_250_CONTOUR_SPACING_M) * GPH_250_CONTOUR_SPACING_M
)
gph_250_vmax = float(
    np.ceil(float(gph_250_frames.max().values) / GPH_250_CONTOUR_SPACING_M) * GPH_250_CONTOUR_SPACING_M
)
gph_250_contour_levels = np.arange(
    gph_250_vmin,
    gph_250_vmax + GPH_250_CONTOUR_SPACING_M,
    GPH_250_CONTOUR_SPACING_M,
)

for i, row in frame_rows.iterrows():
    record = records[i]
    frame_ts = pd.Timestamp(row["valid_time"]).round("h")
    frame_step = row.get("step_hour", np.nan)

    vv_frame = vv_frames_box.isel(frame=i)
    gph_frame = gph_250_frames.isel(frame=i)

    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])
    long_end_a = np.asarray(record["long_end_a"], dtype=float)
    long_end_b = np.asarray(record["long_end_b"], dtype=float)

    fig = plt.figure(figsize=(14, 7))
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

    mesh = ax.pcolormesh(
        vv_frame["longitude"],
        vv_frame["latitude"],
        vv_frame,
        cmap=vv_cmap,
        norm=vv_norm,
        transform=ccrs.PlateCarree(),
        shading="auto",
        zorder=1,
    )

    contour_250 = ax.contour(
        gph_frame["longitude"],
        gph_frame["latitude"],
        gph_frame,
        levels=gph_250_contour_levels,
        colors=GPH_250_CONTOUR_COLOR,
        linewidths=0.9,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    if len(contour_250.levels) > 0:
        ax.clabel(contour_250, contour_250.levels[::2], fmt="%.0f", inline=True, fontsize=7)

    (between_extrema_handle,) = ax.plot(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        color="magenta",
        linewidth=2.0,
        linestyle="-",
        transform=ccrs.PlateCarree(),
        zorder=9,
        label="Line between extrema stops",
    )

    extrema_endpoints_handle = ax.scatter(
        [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
        [float(long_end_a[1]), float(long_end_b[1])],
        s=42,
        c="magenta",
        marker="x",
        linewidths=1.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Extrema stops",
    )

    (traj_line_handle,) = ax.plot(
        line_lons,
        line_lats,
        color="white",
        linewidth=1.8,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label="Backward trajectory (72h to 0h)",
    )

    traj_points_handle = ax.scatter(
        line_lons,
        line_lats,
        s=18,
        c="black",
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=6,
        label=TRAJECTORY_POINT_LABEL,
    )

    current_handle = ax.scatter(
        [lon_curr],
        [lat_curr],
        marker="x",
        s=130,
        c="deepskyblue",
        linewidths=2.3,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label=f"Current {extrema_context['frame_step_hours']}h point",
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    step_text = (
        f"t-{int(frame_step)}h"
        if pd.notna(frame_step)
        else f"{extrema_context['frame_step_hours']}h sample"
    )
    ax.set_title(
        f"250 hPa vertical velocity (box-mean {box_size}x{box_size}, discrete, symmetric) + "
        f"250 hPa GPH contours ({contour_spacing_label} m spacing) | "
        f"{frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
    )

    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=vv_levels,
        orientation="horizontal",
        pad=0.05,
        shrink=0.9,
    )
    cbar.set_label(
        "Vertical velocity (Pa s$^{-1}$): negative=upward motion (ascent), "
        "positive=downward motion (subsidence)"
    )

    contour_handle = Line2D([0], [0], color=GPH_250_CONTOUR_COLOR, lw=0.9)
    ax.legend(
        [
            contour_handle,
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ],
        [
            f"250 hPa GPH contours ({contour_spacing_label} m)",
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {extrema_context['frame_step_hours']}h point",
            "Line between extrema stops",
            "Extrema stops",
        ],
        loc="lower left",
    )
    plt.tight_layout()
    plt.show()


In [ ]:
if "trajectory_df" not in globals():
    raise RuntimeError("Run the backward trajectory cell first so trajectory_df exists.")
if "gph_925_m_all" not in globals():
    raise RuntimeError("Run the shared GPH setup cell first so gph_925_m_all exists.")
required_helpers = [
    "_trace_gradient_path",
]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the helper cell first so these functions exist: " + ", ".join(missing_helpers)
    )
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from pathlib import Path as FilePath

# Rendering controls
FRAME_STEP_HOURS = 6
DIVERGENCE_PRESSURE_LEVELS_HPA = [250.0]
DIVERGENCE_BIN_WIDTH_S1 = 1.0e-6
BOX_SIZE = 25
GPH_CONTOUR_SPACING_M = 40.0
GPH_CONTOUR_COLOR = "black"
DIVERGENCE_FILE = "era5_2021-nov_250-500-925_divergence_vertical_velocity.nc"
DIVERGENCE_PATH_CANDIDATES = [
    FilePath("../data") / DIVERGENCE_FILE,  # when kernel cwd is scripts/
    FilePath("data") / DIVERGENCE_FILE,     # when kernel cwd is repo root
]

divergence_path = next((p for p in DIVERGENCE_PATH_CANDIDATES if p.exists()), None)
if divergence_path is None:
    checked = ", ".join(str(p) for p in DIVERGENCE_PATH_CANDIDATES)
    raise FileNotFoundError(f"Could not find {DIVERGENCE_FILE}. Checked: {checked}")

divergence_ds = xr.open_dataset(divergence_path)
time_coord = "valid_time" if "valid_time" in divergence_ds.coords else "time"
if time_coord not in divergence_ds.coords:
    raise RuntimeError("Expected a valid_time/time coordinate in divergence dataset.")
if "pressure_level" not in divergence_ds.coords:
    raise RuntimeError("Expected pressure_level coordinate in divergence dataset.")
if "d" not in divergence_ds.data_vars:
    raise RuntimeError("Expected variable 'd' (divergence) in divergence dataset.")

box_size = int(BOX_SIZE)
if box_size < 1:
    raise RuntimeError("BOX_SIZE must be >= 1")
if box_size % 2 == 0:
    raise RuntimeError("BOX_SIZE must be odd so the box is centered on each pixel.")

# Same extent as the second-last (gradient/ghost) cell.
extent_lon_min = float(globals().get("region_lon_min", globals().get("GPH_REGION_LON_MIN", 120.0)))
extent_lon_max = float(globals().get("region_lon_max", globals().get("GPH_REGION_LON_MAX", 265.0)))
extent_lat_min = float(globals().get("region_lat_min", 23.0))
extent_lat_max = float(globals().get("region_lat_max", globals().get("GPH_REGION_LAT_MAX", 72.0)))

# _trace_gradient_path uses these globals for domain checks; set them explicitly.
region_lon_min = extent_lon_min
region_lon_max = extent_lon_max
region_lat_min = extent_lat_min
region_lat_max = extent_lat_max

lat_values = divergence_ds["latitude"].values
lat_slice = (
    slice(extent_lat_max, extent_lat_min)
    if float(lat_values[0]) > float(lat_values[-1])
    else slice(extent_lat_min, extent_lat_max)
)

traj = trajectory_df.copy()
traj["valid_time"] = pd.to_datetime(traj["valid_time"])
traj["longitude_360"] = traj["longitude"] % 360.0

if "step_hour" in traj.columns:
    frame_rows = (
        traj.loc[traj["step_hour"] % FRAME_STEP_HOURS == 0]
        .sort_values("step_hour", ascending=False)
        .reset_index(drop=True)
    )
    trajectory_line = traj.sort_values("step_hour", ascending=False).copy()
else:
    # Fallback if step_hour is missing: sample every FRAME_STEP_HOURS rows.
    trajectory_line = traj.sort_values("valid_time", ascending=True).copy()
    frame_rows = trajectory_line.iloc[::FRAME_STEP_HOURS].copy().reset_index(drop=True)
    frame_rows["step_hour"] = np.nan

if frame_rows.empty:
    raise RuntimeError("No 6-hour trajectory points found to plot.")

# Pre-slice divergence domain + pressure once.
div_region_all = divergence_ds["d"].sel(
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
)
div_region_all = div_region_all.sel(pressure_level=DIVERGENCE_PRESSURE_LEVELS_HPA, method="nearest")

# Frame times used for both divergence and 925-hPa GPH selections.
frame_times = pd.to_datetime(frame_rows["valid_time"]).dt.round("h")
frame_times_da = xr.DataArray(frame_times.to_numpy(dtype="datetime64[ns]"), dims="frame")

# Vectorized time selection (faster than per-frame .sel).
div_frames_all = div_region_all.sel({time_coord: frame_times_da}, method="nearest").load()

# BOX_SIZE x BOX_SIZE centered mean for each pixel (250 hPa cell).
div_frames_all_box = div_frames_all.rolling(
    latitude=box_size,
    longitude=box_size,
    center=True,
    min_periods=1,
).mean()

# Matching 925-hPa GPH frames for contouring and extrema tracing.
gph_region = gph_925_m_all.sel(
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
)
gph_frames = gph_region.sel(valid_time=frame_times_da, method="nearest")

# Discrete symmetric divergence buckets (negative=convergence, positive=divergence).
bin_width_s1 = float(DIVERGENCE_BIN_WIDTH_S1)
if not np.isfinite(bin_width_s1) or bin_width_s1 <= 0.0:
    raise RuntimeError("DIVERGENCE_BIN_WIDTH_S1 must be a positive finite value.")

max_abs = float(np.nanmax(np.abs(div_frames_all_box.values)))
if not np.isfinite(max_abs):
    raise RuntimeError("Non-finite divergence range encountered after box averaging.")
if max_abs < bin_width_s1:
    max_abs = bin_width_s1
vmax = float(np.ceil(max_abs / bin_width_s1) * bin_width_s1)
vmin = -vmax
divergence_levels = np.arange(vmin, vmax + bin_width_s1, bin_width_s1)
divergence_nbins = max(len(divergence_levels) - 1, 1)
divergence_cmap = plt.get_cmap("RdBu_r", divergence_nbins)
divergence_norm = mcolors.BoundaryNorm(divergence_levels, ncolors=divergence_nbins, clip=True)

contour_spacing_m = float(GPH_CONTOUR_SPACING_M)
if not np.isfinite(contour_spacing_m) or contour_spacing_m <= 0.0:
    raise RuntimeError("GPH_CONTOUR_SPACING_M must be a positive finite value.")

contour_spacing_label = (
    f"{int(contour_spacing_m)}"
    if float(contour_spacing_m).is_integer()
    else f"{contour_spacing_m:g}"
)
gph_vmin = float(np.floor(float(gph_frames.min().values) / contour_spacing_m) * contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames.max().values) / contour_spacing_m) * contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + contour_spacing_m, contour_spacing_m)

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

# Plot 6-hour trajectory frames at 250 hPa only.
for target_plev_hpa in DIVERGENCE_PRESSURE_LEVELS_HPA:
    div_frames = div_frames_all_box.sel(pressure_level=target_plev_hpa, method="nearest")
    for i, row in frame_rows.iterrows():
        frame_ts = pd.Timestamp(row["valid_time"]).round("h")
        frame_step = row.get("step_hour", np.nan)

        divergence_frame = div_frames.isel(frame=i)
        gph_frame = gph_frames.isel(frame=i)

        lon_curr = float(row["longitude_360"])
        lat_curr = float(row["latitude"])

        # Find local decreasing/increasing extrema branches from the current trajectory point.
        dec_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="decrease",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )
        inc_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="increase",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )

        long_end_a = np.array([dec_trace["final_lon"], dec_trace["final_lat"]], dtype=float)
        long_end_b = np.array([inc_trace["final_lon"], inc_trace["final_lat"]], dtype=float)

        fig = plt.figure(figsize=(14, 7))
        ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

        ax.coastlines(resolution="110m", linewidth=0.8, color="black")
        ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

        mesh = ax.pcolormesh(
            divergence_frame["longitude"],
            divergence_frame["latitude"],
            divergence_frame,
            cmap=divergence_cmap,
            norm=divergence_norm,
            transform=ccrs.PlateCarree(),
            shading="auto",
            zorder=1,
        )

        contour_gph = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=gph_contour_levels,
            colors=GPH_CONTOUR_COLOR,
            linewidths=0.9,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        if len(contour_gph.levels) > 0:
            ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)
        contour_handle_for_legend = Line2D([0], [0], color=GPH_CONTOUR_COLOR, lw=0.9)

        (between_extrema_handle,) = ax.plot(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            color="magenta",
            linewidth=2.0,
            linestyle="-",
            transform=ccrs.PlateCarree(),
            zorder=9,
            label="Line between extrema stops",
        )

        extrema_endpoints_handle = ax.scatter(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            s=42,
            c="magenta",
            marker="x",
            linewidths=1.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label="Extrema stops",
        )

        (traj_line_handle,) = ax.plot(
            line_lons,
            line_lats,
            color="white",
            linewidth=1.8,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label="Backward trajectory (72h to 0h)",
        )

        traj_points_handle = ax.scatter(
            line_lons,
            line_lats,
            s=18,
            c="black",
            alpha=0.35,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label=TRAJECTORY_POINT_LABEL,
        )

        current_handle = ax.scatter(
            [lon_curr],
            [lat_curr],
            marker="x",
            s=130,
            c="deepskyblue",
            linewidths=2.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label=f"Current {FRAME_STEP_HOURS}h point",
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False

        plotted_plev = int(np.round(float(divergence_frame["pressure_level"].values)))
        step_text = f"t-{int(frame_step)}h" if pd.notna(frame_step) else f"{FRAME_STEP_HOURS}h sample"
        ax.set_title(
            f"{plotted_plev} hPa Divergence (box-mean {box_size}x{box_size}, discrete, symmetric) + 925 hPa GPH contours ({contour_spacing_label} m spacing) "
            f"| {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
        )

        cbar = plt.colorbar(
            mesh,
            ax=ax,
            boundaries=divergence_levels,
            orientation="horizontal",
            pad=0.05,
            shrink=0.9,
        )
        cbar.set_label(
            "Divergence (s$^{-1}$): positive=divergence (spreading), "
            "negative=convergence (concentrating)"
        )

        legend_handles = [
            contour_handle_for_legend,
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ]
        legend_labels = [
            f"925 hPa GPH contours ({contour_spacing_label} m)",
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {FRAME_STEP_HOURS}h point",
            "Line between extrema stops",
            "Extrema stops",
        ]

        ax.legend(legend_handles, legend_labels, loc="lower left")
        plt.tight_layout()
        plt.show()


In [ ]:
if "trajectory_df" not in globals():
    raise RuntimeError("Run the backward trajectory cell first so trajectory_df exists.")
if "gph_925_m_all" not in globals():
    raise RuntimeError("Run the shared GPH setup cell first so gph_925_m_all exists.")
required_helpers = [
    "_trace_gradient_path",
]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the helper cell first so these functions exist: " + ", ".join(missing_helpers)
    )
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from pathlib import Path as FilePath

# Rendering controls
FRAME_STEP_HOURS = 6
DIVERGENCE_PRESSURE_LEVELS_HPA = [500.0]
DIVERGENCE_BIN_WIDTH_S1 = 1.0e-6
BOX_SIZE = 25
GPH_CONTOUR_SPACING_M = 40.0
GPH_CONTOUR_COLOR = "black"
DIVERGENCE_FILE = "era5_2021-nov_250-500-925_divergence_vertical_velocity.nc"
DIVERGENCE_PATH_CANDIDATES = [
    FilePath("../data") / DIVERGENCE_FILE,  # when kernel cwd is scripts/
    FilePath("data") / DIVERGENCE_FILE,     # when kernel cwd is repo root
]

divergence_path = next((p for p in DIVERGENCE_PATH_CANDIDATES if p.exists()), None)
if divergence_path is None:
    checked = ", ".join(str(p) for p in DIVERGENCE_PATH_CANDIDATES)
    raise FileNotFoundError(f"Could not find {DIVERGENCE_FILE}. Checked: {checked}")

divergence_ds = xr.open_dataset(divergence_path)
time_coord = "valid_time" if "valid_time" in divergence_ds.coords else "time"
if time_coord not in divergence_ds.coords:
    raise RuntimeError("Expected a valid_time/time coordinate in divergence dataset.")
if "pressure_level" not in divergence_ds.coords:
    raise RuntimeError("Expected pressure_level coordinate in divergence dataset.")
if "d" not in divergence_ds.data_vars:
    raise RuntimeError("Expected variable 'd' (divergence) in divergence dataset.")

box_size = int(BOX_SIZE)
if box_size < 1:
    raise RuntimeError("BOX_SIZE must be >= 1")
if box_size % 2 == 0:
    raise RuntimeError("BOX_SIZE must be odd so the box is centered on each pixel.")

# Same extent as the second-last (gradient/ghost) cell.
extent_lon_min = float(globals().get("region_lon_min", globals().get("GPH_REGION_LON_MIN", 120.0)))
extent_lon_max = float(globals().get("region_lon_max", globals().get("GPH_REGION_LON_MAX", 265.0)))
extent_lat_min = float(globals().get("region_lat_min", 23.0))
extent_lat_max = float(globals().get("region_lat_max", globals().get("GPH_REGION_LAT_MAX", 72.0)))

# _trace_gradient_path uses these globals for domain checks; set them explicitly.
region_lon_min = extent_lon_min
region_lon_max = extent_lon_max
region_lat_min = extent_lat_min
region_lat_max = extent_lat_max

lat_values = divergence_ds["latitude"].values
lat_slice = (
    slice(extent_lat_max, extent_lat_min)
    if float(lat_values[0]) > float(lat_values[-1])
    else slice(extent_lat_min, extent_lat_max)
)

traj = trajectory_df.copy()
traj["valid_time"] = pd.to_datetime(traj["valid_time"])
traj["longitude_360"] = traj["longitude"] % 360.0

if "step_hour" in traj.columns:
    frame_rows = (
        traj.loc[traj["step_hour"] % FRAME_STEP_HOURS == 0]
        .sort_values("step_hour", ascending=False)
        .reset_index(drop=True)
    )
    trajectory_line = traj.sort_values("step_hour", ascending=False).copy()
else:
    # Fallback if step_hour is missing: sample every FRAME_STEP_HOURS rows.
    trajectory_line = traj.sort_values("valid_time", ascending=True).copy()
    frame_rows = trajectory_line.iloc[::FRAME_STEP_HOURS].copy().reset_index(drop=True)
    frame_rows["step_hour"] = np.nan

if frame_rows.empty:
    raise RuntimeError("No 6-hour trajectory points found to plot.")

# Pre-slice divergence domain + pressure once.
div_region_all = divergence_ds["d"].sel(
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
)
div_region_all = div_region_all.sel(pressure_level=DIVERGENCE_PRESSURE_LEVELS_HPA, method="nearest")

# Frame times used for both divergence and 925-hPa GPH selections.
frame_times = pd.to_datetime(frame_rows["valid_time"]).dt.round("h")
frame_times_da = xr.DataArray(frame_times.to_numpy(dtype="datetime64[ns]"), dims="frame")

# Vectorized time selection (faster than per-frame .sel).
div_frames_all = div_region_all.sel({time_coord: frame_times_da}, method="nearest").load()

# BOX_SIZE x BOX_SIZE centered mean for each pixel (500 hPa cell).
div_frames_all_box = div_frames_all.rolling(
    latitude=box_size,
    longitude=box_size,
    center=True,
    min_periods=1,
).mean()

# Matching 925-hPa GPH frames for contouring and extrema tracing.
gph_region = gph_925_m_all.sel(
    longitude=slice(extent_lon_min, extent_lon_max),
    latitude=lat_slice,
)
gph_frames = gph_region.sel(valid_time=frame_times_da, method="nearest")

# Discrete symmetric divergence buckets (negative=convergence, positive=divergence).
bin_width_s1 = float(DIVERGENCE_BIN_WIDTH_S1)
if not np.isfinite(bin_width_s1) or bin_width_s1 <= 0.0:
    raise RuntimeError("DIVERGENCE_BIN_WIDTH_S1 must be a positive finite value.")

max_abs = float(np.nanmax(np.abs(div_frames_all_box.values)))
if not np.isfinite(max_abs):
    raise RuntimeError("Non-finite divergence range encountered after box averaging.")
if max_abs < bin_width_s1:
    max_abs = bin_width_s1
vmax = float(np.ceil(max_abs / bin_width_s1) * bin_width_s1)
vmin = -vmax
divergence_levels = np.arange(vmin, vmax + bin_width_s1, bin_width_s1)
divergence_nbins = max(len(divergence_levels) - 1, 1)
divergence_cmap = plt.get_cmap("RdBu_r", divergence_nbins)
divergence_norm = mcolors.BoundaryNorm(divergence_levels, ncolors=divergence_nbins, clip=True)

contour_spacing_m = float(GPH_CONTOUR_SPACING_M)
if not np.isfinite(contour_spacing_m) or contour_spacing_m <= 0.0:
    raise RuntimeError("GPH_CONTOUR_SPACING_M must be a positive finite value.")

contour_spacing_label = (
    f"{int(contour_spacing_m)}"
    if float(contour_spacing_m).is_integer()
    else f"{contour_spacing_m:g}"
)
gph_vmin = float(np.floor(float(gph_frames.min().values) / contour_spacing_m) * contour_spacing_m)
gph_vmax = float(np.ceil(float(gph_frames.max().values) / contour_spacing_m) * contour_spacing_m)
gph_contour_levels = np.arange(gph_vmin, gph_vmax + contour_spacing_m, contour_spacing_m)

line_lons = trajectory_line["longitude_360"].to_numpy(dtype=float)
line_lats = trajectory_line["latitude"].to_numpy(dtype=float)

# Plot 6-hour trajectory frames at 500 hPa only.
for target_plev_hpa in DIVERGENCE_PRESSURE_LEVELS_HPA:
    div_frames = div_frames_all_box.sel(pressure_level=target_plev_hpa, method="nearest")
    for i, row in frame_rows.iterrows():
        frame_ts = pd.Timestamp(row["valid_time"]).round("h")
        frame_step = row.get("step_hour", np.nan)

        divergence_frame = div_frames.isel(frame=i)
        gph_frame = gph_frames.isel(frame=i)

        lon_curr = float(row["longitude_360"])
        lat_curr = float(row["latitude"])

        # Find local decreasing/increasing extrema branches from the current trajectory point.
        dec_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="decrease",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )
        inc_trace = _trace_gradient_path(
            gph_frame,
            lon_curr,
            lat_curr,
            prefer="increase",
            step_km=GRAD_STEP_KM,
            probe_deg=GRAD_PROBE_DEG,
            grad_min_mag=GRAD_MIN_MAG_M_PER_KM,
            max_steps=GRAD_MAX_STEPS,
            monotonic_tol=GRAD_MONOTONIC_TOL_M,
        )

        long_end_a = np.array([dec_trace["final_lon"], dec_trace["final_lat"]], dtype=float)
        long_end_b = np.array([inc_trace["final_lon"], inc_trace["final_lat"]], dtype=float)

        fig = plt.figure(figsize=(14, 7))
        ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent([extent_lon_min, extent_lon_max, extent_lat_min, extent_lat_max], crs=ccrs.PlateCarree())

        ax.coastlines(resolution="110m", linewidth=0.8, color="black")
        ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="black")

        mesh = ax.pcolormesh(
            divergence_frame["longitude"],
            divergence_frame["latitude"],
            divergence_frame,
            cmap=divergence_cmap,
            norm=divergence_norm,
            transform=ccrs.PlateCarree(),
            shading="auto",
            zorder=1,
        )

        contour_gph = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=gph_contour_levels,
            colors=GPH_CONTOUR_COLOR,
            linewidths=0.9,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        if len(contour_gph.levels) > 0:
            ax.clabel(contour_gph, contour_gph.levels[::2], fmt="%.0f", inline=True, fontsize=7)
        contour_handle_for_legend = Line2D([0], [0], color=GPH_CONTOUR_COLOR, lw=0.9)

        (between_extrema_handle,) = ax.plot(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            color="magenta",
            linewidth=2.0,
            linestyle="-",
            transform=ccrs.PlateCarree(),
            zorder=9,
            label="Line between extrema stops",
        )

        extrema_endpoints_handle = ax.scatter(
            [float(long_end_a[0]) % 360.0, float(long_end_b[0]) % 360.0],
            [float(long_end_a[1]), float(long_end_b[1])],
            s=42,
            c="magenta",
            marker="x",
            linewidths=1.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label="Extrema stops",
        )

        (traj_line_handle,) = ax.plot(
            line_lons,
            line_lats,
            color="white",
            linewidth=1.8,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label="Backward trajectory (72h to 0h)",
        )

        traj_points_handle = ax.scatter(
            line_lons,
            line_lats,
            s=18,
            c="black",
            alpha=0.35,
            transform=ccrs.PlateCarree(),
            zorder=6,
            label=TRAJECTORY_POINT_LABEL,
        )

        current_handle = ax.scatter(
            [lon_curr],
            [lat_curr],
            marker="x",
            s=130,
            c="deepskyblue",
            linewidths=2.3,
            transform=ccrs.PlateCarree(),
            zorder=10,
            label=f"Current {FRAME_STEP_HOURS}h point",
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False

        plotted_plev = int(np.round(float(divergence_frame["pressure_level"].values)))
        step_text = f"t-{int(frame_step)}h" if pd.notna(frame_step) else f"{FRAME_STEP_HOURS}h sample"
        ax.set_title(
            f"{plotted_plev} hPa Divergence (box-mean {box_size}x{box_size}, discrete, symmetric) + 925 hPa GPH contours ({contour_spacing_label} m spacing) "
            f"| {frame_ts.strftime('%Y-%m-%d %H:%M UTC')} ({step_text})"
        )

        cbar = plt.colorbar(
            mesh,
            ax=ax,
            boundaries=divergence_levels,
            orientation="horizontal",
            pad=0.05,
            shrink=0.9,
        )
        cbar.set_label(
            "Divergence (s$^{-1}$): positive=divergence (spreading), "
            "negative=convergence (concentrating)"
        )

        legend_handles = [
            contour_handle_for_legend,
            traj_line_handle,
            traj_points_handle,
            current_handle,
            between_extrema_handle,
            extrema_endpoints_handle,
        ]
        legend_labels = [
            f"925 hPa GPH contours ({contour_spacing_label} m)",
            "Backward trajectory (72h to 0h)",
            TRAJECTORY_POINT_LABEL,
            f"Current {FRAME_STEP_HOURS}h point",
            "Line between extrema stops",
            "Extrema stops",
        ]

        ax.legend(legend_handles, legend_labels, loc="lower left")
        plt.tight_layout()
        plt.show()
